In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import shutil

In [2]:
orig_annot_path = Path("/media/MX500/valentin/datasets/SNP_Summer_2023/SNP_Summer_2023_annotated_tracks_v4")
annot_path = Path("/media/EVO870/datasets/prompting-mammalps-v2/annotations")
annot_json_files = [f for f in list(annot_path.rglob("*.json")) if "E" in f.stem] 
species_df = pd.read_csv("/media/MX500/valentin/datasets/SNP_Summer_2023/annotation/tublets_day_split_species_v4_.csv", index_col=0)

output_annotation_path = Path("/media/EVO870/datasets/prompting-mammalps-v2/annotations_orig_2023")

In [3]:
def from_prop_to_abs_coords(xywh_prop, width, height):
    if isinstance(xywh_prop[0], int): 
        # Has already been converted
        # Happens if the notebook is ran multiple times
        return xywh_prop

    xyxy_coord = [
        xywh_prop[0] * width,
        xywh_prop[1] * height,
        (xywh_prop[0] + xywh_prop[2]) * width,
        (xywh_prop[1] + xywh_prop[3]) * height
    ]
    xyxy_coord = [int(np.round(c)) for c in xyxy_coord]

    return xyxy_coord

In [ ]:
for annot_json_file in tqdm(annot_json_files):
    output_json_file = output_annotation_path / annot_json_file.relative_to(annot_path)

    #Find corresponding file in original annotation folder
    try:
        orig_annot_file = next(orig_annot_path.rglob(f"*{annot_json_file.stem}*"))
        with open(orig_annot_file, "r") as f:
            orig_content = json.load(f)
        
        for orig_frame in orig_content["frames"]:
            orig_frame.pop("max_detection_conf", None)
            valid_detections = []
            for detection in [d for d in orig_frame["detections"] if "attributes" in d]:
                detection.pop("conf", None)
                detection.pop("category", None)
                detection.pop("occluded", None)
                detection["bbox"] = from_prop_to_abs_coords(detection["bbox"], 1920, 1080)
                detection["attributes"].pop("red_deer_age_group", None)
                detection["attributes"].pop("red_deer_adult_sex", None)
                
                # Add species label
                track_id = detection["track_id"]
                try:
                    species = (
                        species_df[
                            (species_df["file_id"] == annot_json_file.stem) & 
                            (species_df["track_id"] == track_id)]
                        ["species"].values[0]
                        )
                    detection["attributes"]["Species"] = species
                    # Match label space
                    if detection["attributes"]["Activity"] == "none":
                        detection["attributes"]["Activity"] = "unknown"
                    if detection["attributes"]["Activity"] == "marking":
                        detection["attributes"]["Activity"] = "marking_or_wallowing"
                    if detection["attributes"]["Action"] == "laying":
                        detection["attributes"]["Action"] = "laying_down"
                    if detection["attributes"]["Action"] == "scratching_hoof":
                        detection["attributes"]["Action"] = "pawing_ground"
                    if detection["attributes"]["Action"] == "scratching_antlers":
                        detection["attributes"]["Action"] = "rubbing_antlers_on_ground"
                    if detection["attributes"]["Action"] == "scratching_body":
                        detection["attributes"]["Action"] = "scratching_own_head_or_body"
                    if detection["attributes"]["Action"] == "shaking_fur":
                        detection["attributes"]["Action"] = "shaking_head_or_fur"
                    if detection["attributes"]["Action"] == "running":
                        detection["attributes"]["Action"] = "trotting_or_running"
                    if detection["attributes"]["Action2"] == "laying":
                        detection["attributes"]["Action2"] = "laying_down"
                    if detection["attributes"]["Action2"] == "scratching_hoof":
                        detection["attributes"]["Action2"] = "pawing_ground"
                    if detection["attributes"]["Action2"] == "scratching_antlers":
                        detection["attributes"]["Action2"] = "rubbing_antlers_on_ground"
                    if detection["attributes"]["Action2"] == "scratching_body":
                        detection["attributes"]["Action2"] = "scratching_own_head_or_body"
                    if detection["attributes"]["Action2"] == "shaking_fur":
                        detection["attributes"]["Action2"] = "shaking_head_or_fur"
                    if detection["attributes"]["Action2"] == "running":
                        detection["attributes"]["Action2"] = "trotting_or_running"
                    valid_detections.append(detection)
                except IndexError:
                    pass
                except KeyError:
                    pass
            orig_frame["detections"] = valid_detections

        # Use info from annotated file, which also contains the weather label
        with open(annot_json_file, "r") as f:
            info_section = json.load(f)["info"]
        orig_content["info"] = info_section

        # Save to output folder
        os.makedirs(output_json_file.parent, exist_ok=True)
        with open(output_json_file, "w") as f:
            json.dump(orig_content, f, indent=2)

    except StopIteration:
        print(f"Could not find annotation for {annot_json_file.stem}, using default one")
        # Save the same file
        os.makedirs(output_json_file.parent, exist_ok=True)
        shutil.copy(annot_json_file, output_json_file)

  0%|          | 1/1738 [00:00<06:59,  4.14it/s]

Could not find annotation for S3_C3_E523_V0324, using default one


  1%|          | 10/1738 [00:08<33:51,  1.18s/it]

Could not find annotation for S3_C3_E506_V0291, using default one


  1%|          | 14/1738 [00:12<31:07,  1.08s/it]

...

 69%|██████▉   | 1196/1738 [06:14<02:55,  3.09it/s]

Could not find annotation for S1_C1_E227_V0864, using default one
Could not find annotation for S1_C1_E221_V0861, using default one


 97%|█████████▋| 1687/1738 [08:48<00:11,  4.39it/s]

Could not find annotation for S2_C1_E321_V0067, using default one


100%|██████████| 1738/1738 [09:13<00:00,  3.14it/s]


In [4]:
for annot_json_file in tqdm(annot_json_files):
    output_json_file = ("/media/EVO870/datasets/prompting-mammalps-v2/annotations_2023/") / annot_json_file.relative_to(annot_path)
    with open(annot_json_file, "r") as f:
        annot_dict = json.load(f)
    
    for frame_annot in annot_dict["frames"]:
        for detection in frame_annot["detections"]:
            if "attributes" in detection:
                detection["attributes"].pop("Deer_adult_sex", None)
                detection["attributes"].pop("Deer_age", None)
    
    os.makedirs(output_json_file.parent, exist_ok=True)
    with open(output_json_file, "w") as f:
        json.dump(annot_dict, f, indent=2)

100%|██████████| 1738/1738 [00:25<00:00, 68.95it/s] 
